# Section C — AOI Selection: Where Burns Most in California?

**Goal:** Identify the best area of interest (AOI) for OpenFire's ML pipeline.

**Criteria for a good AOI:**
- High fire frequency (2000–2025) — gives us plenty of training labels
- Active in Sentinel-2 era (2015+) — so we have satellite imagery
- Manageable geographic scope — one or a few counties
- Mix of fire sizes — not just one mega-fire dominating

**Approach:** Rasterize fire perimeters onto a grid, count burns per cell,
then aggregate by county to rank candidate AOIs.

In [ ]:
# C1: Filter to 2000+ fires
gdf_modern = gdf[gdf["year"] >= 2000].copy()
gdf_s2 = gdf[gdf["year"] >= 2015].copy()

print(f"All fires:       {len(gdf):,}")
print(f"Since 2000:      {len(gdf_modern):,} fires, {gdf_modern['acres'].sum():,.0f} acres")
print(f"Since 2015 (S2): {len(gdf_s2):,} fires, {gdf_s2['acres'].sum():,.0f} acres")
print(f"\nFires per year (2000+):")
print(gdf_modern.groupby("year").size().to_string())

In [ ]:
# C2: Rasterize — count how many times each grid cell burned (2000+)
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize

# Use California bounds from our boundary file
ca_bounds = ca_boundary.total_bounds  # [minx, miny, maxx, maxy]
res = RASTER_RESOLUTION_M

# Compute grid dimensions
width = int((ca_bounds[2] - ca_bounds[0]) / res)
height = int((ca_bounds[3] - ca_bounds[1]) / res)
transform = from_bounds(*ca_bounds, width, height)

print(f"Grid: {width} x {height} = {width * height:,} cells at {res}m resolution")

# Rasterize: for each fire, burn a 1 into its footprint, then sum
burn_count = np.zeros((height, width), dtype=np.int16)

for _, row in tqdm(gdf_modern.iterrows(), total=len(gdf_modern), desc="Rasterizing 2000+ fires"):
    if row.geometry is None or row.geometry.is_empty:
        continue
    try:
        raster = rasterize(
            [(row.geometry, 1)],
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype="uint8",
        )
        burn_count += raster
    except Exception:
        continue

print(f"\nBurn count stats:")
print(f"  Max burns in one cell: {burn_count.max()}")
print(f"  Cells burned at least once: {(burn_count >= 1).sum():,}")
print(f"  Cells burned 2+ times: {(burn_count >= 2).sum():,}")
print(f"  Cells burned 3+ times: {(burn_count >= 3).sum():,}")

In [ ]:
# C3: Same thing for Sentinel-2 era (2015+)
burn_count_s2 = np.zeros((height, width), dtype=np.int16)

for _, row in tqdm(gdf_s2.iterrows(), total=len(gdf_s2), desc="Rasterizing 2015+ fires"):
    if row.geometry is None or row.geometry.is_empty:
        continue
    try:
        raster = rasterize(
            [(row.geometry, 1)],
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype="uint8",
        )
        burn_count_s2 += raster
    except Exception:
        continue

print(f"\nBurn count stats (Sentinel-2 era, 2015+):")
print(f"  Max burns in one cell: {burn_count_s2.max()}")
print(f"  Cells burned at least once: {(burn_count_s2 >= 1).sum():,}")
print(f"  Cells burned 2+ times: {(burn_count_s2 >= 2).sum():,}")

In [ ]:
# C4: Heatmap — burn frequency since 2000
from matplotlib.colors import BoundaryNorm, ListedColormap

fig, axes = plt.subplots(1, 2, figsize=(18, 12))

# Custom colormap: 0=transparent, 1=yellow, 2=orange, 3+=red
cmap_colors = ["#0a0a0a", "#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"]
cmap = ListedColormap(cmap_colors)
bounds_cm = [0, 0.5, 1.5, 2.5, 3.5, 4.5, 20]
norm = BoundaryNorm(bounds_cm, cmap.N)

for ax, data, title in [
    (axes[0], burn_count, "Burn Frequency 2000–2025"),
    (axes[1], burn_count_s2, "Burn Frequency 2015–2025 (Sentinel-2 era)"),
]:
    ax.set_facecolor(BG_COLOR)
    ca_boundary.boundary.plot(ax=ax, color="#444444", linewidth=0.8)
    
    # Convert raster to map coordinates for imshow
    extent = [ca_bounds[0], ca_bounds[2], ca_bounds[1], ca_bounds[3]]
    im = ax.imshow(
        data, extent=extent, origin="lower",
        cmap=cmap, norm=norm, interpolation="nearest",
    )
    
    ax.set_title(title, fontsize=14, fontweight="bold", color="white", pad=10)
    ax.set_axis_off()
    ax.set_aspect("equal")

# Colorbar
cbar = fig.colorbar(im, ax=axes, orientation="horizontal", fraction=0.03, pad=0.04)
cbar.set_ticks([0.25, 1, 2, 3, 4, 5])
cbar.set_ticklabels(["0", "1", "2", "3", "4", "5+"])
cbar.set_label("Number of times burned", fontsize=12)

fig.patch.set_facecolor(BG_COLOR)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "burn_frequency_heatmap.png", dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

In [ ]:
# C5: Load county boundaries and aggregate burn stats per county
# Using Census TIGER county boundaries
import urllib.request

county_zip = RAW_DIR / "tl_2023_06_cousub.zip"
county_url = "https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip"

county_path = RAW_DIR / "counties"
county_shp = county_path / "ca_counties.shp"

if not county_shp.exists():
    # Download county boundaries
    county_zip = RAW_DIR / "tl_2023_us_county.zip"
    if not county_zip.exists():
        print("Downloading county boundaries...")
        resp = requests.get(county_url, timeout=60)
        resp.raise_for_status()
        with open(county_zip, "wb") as f:
            f.write(resp.content)
        print(f"✓ Downloaded {county_zip.name}")
    
    # Extract California counties (FIPS state code 06)
    counties_all = gpd.read_file(f"zip://{county_zip}")
    ca_counties = counties_all[counties_all["STATEFP"] == "06"].copy()
    county_path.mkdir(parents=True, exist_ok=True)
    ca_counties.to_file(county_shp)
    print(f"✓ Extracted {len(ca_counties)} California counties")
else:
    ca_counties = gpd.read_file(county_shp)
    print(f"✓ Loaded {len(ca_counties)} California counties")

ca_counties = ca_counties.to_crs(CRS_CA_ALBERS)
print(f"Counties: {', '.join(sorted(ca_counties['NAME'].tolist())[:10])}...")

In [ ]:
# C6: Per-county fire stats — the key ranking table

county_stats = []

for _, county in ca_counties.iterrows():
    county_name = county["NAME"]
    county_geom = county.geometry
    
    # Fires since 2000 that intersect this county
    fires_2000 = gdf_modern[gdf_modern.intersects(county_geom)]
    # Fires since 2015 (Sentinel-2 era)
    fires_s2 = gdf_s2[gdf_s2.intersects(county_geom)]
    
    # Total burned area (clipped to county)
    try:
        burned_2000 = fires_2000.clip(county_geom)
        acres_2000 = burned_2000.geometry.area.sum() / 4046.86  # sq m to acres
    except Exception:
        acres_2000 = fires_2000["acres"].sum()
    
    try:
        burned_s2 = fires_s2.clip(county_geom)
        acres_s2 = burned_s2.geometry.area.sum() / 4046.86
    except Exception:
        acres_s2 = fires_s2["acres"].sum()
    
    county_area_acres = county_geom.area / 4046.86
    
    county_stats.append({
        "county": county_name,
        "fires_2000": len(fires_2000),
        "fires_s2": len(fires_s2),
        "acres_burned_2000": acres_2000,
        "acres_burned_s2": acres_s2,
        "county_acres": county_area_acres,
        "pct_burned_2000": (acres_2000 / county_area_acres * 100) if county_area_acres > 0 else 0,
        "pct_burned_s2": (acres_s2 / county_area_acres * 100) if county_area_acres > 0 else 0,
    })

county_df = pd.DataFrame(county_stats)

# Sort by number of fires in S2 era (what matters most for our ML pipeline)
county_df = county_df.sort_values("fires_s2", ascending=False).reset_index(drop=True)

print("\nTop 15 counties by fire count (Sentinel-2 era, 2015+):")
print("=" * 90)
print(f"{'County':<20} {'Fires S2':>10} {'Acres S2':>12} {'%Burned S2':>10} {'Fires 2000+':>12} {'Acres 2000+':>14}")
print("-" * 90)
for _, row in county_df.head(15).iterrows():
    print(
        f"{row['county']:<20} {row['fires_s2']:>10,} {row['acres_burned_s2']:>12,.0f} "
        f"{row['pct_burned_s2']:>9.1f}% {row['fires_2000']:>12,} {row['acres_burned_2000']:>14,.0f}"
    )

In [ ]:
# C7: Visual — top counties bar chart + map
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Bar chart: top 15 by S2-era fire count
ax = axes[0]
top15 = county_df.head(15)
bars = ax.barh(range(len(top15)), top15["fires_s2"], color="#fd8d3c", alpha=0.8)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15["county"])
ax.set_xlabel("Number of Fires (2015–2025)", fontsize=12)
ax.set_title("Top Counties by Fire Count\n(Sentinel-2 Era)", fontsize=14, fontweight="bold")
ax.invert_yaxis()
ax.set_facecolor("#f5f5f5")

# Add second bar for 2000+ fires in lighter color
ax.barh(range(len(top15)), top15["fires_2000"], color="#bd0026", alpha=0.3)
ax.legend(["2015–2025", "2000–2025"], loc="lower right")

# Map: color counties by S2 fire count
ax = axes[1]
ax.set_facecolor(BG_COLOR)

county_merged = ca_counties.merge(county_df, left_on="NAME", right_on="county", how="left")
county_merged["fires_s2"] = county_merged["fires_s2"].fillna(0)

county_merged.plot(
    ax=ax, column="fires_s2", cmap="YlOrRd", linewidth=0.5,
    edgecolor="#333333", legend=True,
    legend_kwds={"label": "Fire count (2015–2025)", "shrink": 0.6},
)

# Label top 5 counties
for _, row in county_merged.nlargest(5, "fires_s2").iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        row["NAME"], (centroid.x, centroid.y),
        fontsize=8, fontweight="bold", color="white", ha="center",
        path_effects=[pe.withStroke(linewidth=2, foreground="black")],
    )

ax.set_title("Fire Frequency by County\n(Sentinel-2 Era)", fontsize=14, fontweight="bold", color="white")
ax.set_axis_off()

fig.patch.set_facecolor("white")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "county_fire_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# C8: Zoom into top candidate counties — detailed view
# Show the actual fire perimeters within the top 3 counties
import matplotlib.patheffects as pe

top3 = county_df.head(3)["county"].tolist()
print(f"Zooming into top 3 candidates: {top3}")

fig, axes = plt.subplots(1, 3, figsize=(20, 8))

for ax, county_name in zip(axes, top3):
    county_geom = ca_counties[ca_counties["NAME"] == county_name].geometry.iloc[0]
    county_bounds = ca_counties[ca_counties["NAME"] == county_name].total_bounds
    
    # County outline
    ca_counties[ca_counties["NAME"] == county_name].boundary.plot(
        ax=ax, color="#666666", linewidth=1.5
    )
    
    # Fires since 2015 in this county
    fires_here = gdf_s2[gdf_s2.intersects(county_geom)]
    
    if not fires_here.empty:
        fires_here.plot(
            ax=ax, color="#fd8d3c", alpha=0.5, edgecolor="#ff6600", linewidth=0.3
        )
        
        # Label the biggest fires
        for _, fire in fires_here.nlargest(5, "acres").iterrows():
            centroid = fire.geometry.centroid
            label = f"{fire['fire_name']} ({int(fire['year'])})"
            ax.annotate(
                label, (centroid.x, centroid.y), fontsize=6,
                color="white", ha="center", fontweight="bold",
                path_effects=[pe.withStroke(linewidth=1.5, foreground="black")],
            )
    
    stats = county_df[county_df["county"] == county_name].iloc[0]
    ax.set_title(
        f"{county_name} County\n{int(stats['fires_s2'])} fires (S2 era) | "
        f"{stats['acres_burned_s2']:,.0f} acres",
        fontsize=11, fontweight="bold", color="white",
    )
    ax.set_facecolor(BG_COLOR)
    ax.set_xlim(county_bounds[0] - 5000, county_bounds[2] + 5000)
    ax.set_ylim(county_bounds[1] - 5000, county_bounds[3] + 5000)
    ax.set_axis_off()

fig.patch.set_facecolor(BG_COLOR)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top3_county_detail.png", dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

In [ ]:
# C9: Generate GEE-ready bounding box for the chosen AOI
# Change this to whichever county (or counties) you pick

CHOSEN_COUNTIES = [county_df.iloc[0]["county"]]  # Top county by default
# Uncomment below to pick manually:
# CHOSEN_COUNTIES = ["Butte"]  # or ["Butte", "Plumas"] for multi-county

aoi_counties = ca_counties[ca_counties["NAME"].isin(CHOSEN_COUNTIES)]
aoi_geom = aoi_counties.dissolve().geometry.iloc[0]

# Get bounding box in WGS84 (for GEE)
aoi_wgs84 = aoi_counties.to_crs(CRS_WGS84).dissolve()
aoi_bbox = aoi_wgs84.total_bounds  # [minx, miny, maxx, maxy] = [west, south, east, north]

print(f"Chosen AOI: {', '.join(CHOSEN_COUNTIES)} County")
print(f"\nBounding box (WGS84):")
print(f"  West:  {aoi_bbox[0]:.4f}")
print(f"  South: {aoi_bbox[1]:.4f}")
print(f"  East:  {aoi_bbox[2]:.4f}")
print(f"  North: {aoi_bbox[3]:.4f}")

# GEE JavaScript format
print(f"\n// Google Earth Engine (JavaScript):")
print(f"var aoi = ee.Geometry.Rectangle([{aoi_bbox[0]:.4f}, {aoi_bbox[1]:.4f}, {aoi_bbox[2]:.4f}, {aoi_bbox[3]:.4f}]);")

# GEE Python format
print(f"\n# Google Earth Engine (Python):")
print(f"aoi = ee.Geometry.Rectangle([{aoi_bbox[0]:.4f}, {aoi_bbox[1]:.4f}, {aoi_bbox[2]:.4f}, {aoi_bbox[3]:.4f}])")

# Fire summary for chosen AOI
aoi_fires_s2 = gdf_s2[gdf_s2.intersects(aoi_geom)]
aoi_fires_2000 = gdf_modern[gdf_modern.intersects(aoi_geom)]

print(f"\nAOI fire summary:")
print(f"  Fires 2000–2025: {len(aoi_fires_2000):,}")
print(f"  Fires 2015–2025: {len(aoi_fires_s2):,}")
print(f"  Acres 2015–2025: {aoi_fires_s2['acres'].sum():,.0f}")
print(f"\nTop 10 fires in AOI (2015+):")
print(aoi_fires_s2.nlargest(10, 'acres')[['fire_name', 'year', 'acres']].to_string(index=False))

In [ ]:
# C10: Final AOI map
fig, ax = plt.subplots(figsize=(12, 10))
ax.set_facecolor(BG_COLOR)

# All of California in dim
ca_counties.boundary.plot(ax=ax, color="#333333", linewidth=0.3)

# Highlight chosen county
aoi_counties.plot(ax=ax, color="none", edgecolor="#00ff88", linewidth=2.5)

# S2-era fires in the AOI
aoi_fires_s2.plot(ax=ax, color="#fd8d3c", alpha=0.6, linewidth=0)

# Bounding box in EPSG:3310
aoi_bbox_albers = aoi_counties.total_bounds
from matplotlib.patches import Rectangle
rect = Rectangle(
    (aoi_bbox_albers[0], aoi_bbox_albers[1]),
    aoi_bbox_albers[2] - aoi_bbox_albers[0],
    aoi_bbox_albers[3] - aoi_bbox_albers[1],
    linewidth=1.5, edgecolor="#00ff88", facecolor="none", linestyle="--",
)
ax.add_patch(rect)

ax.set_title(
    f"OpenFire AOI: {', '.join(CHOSEN_COUNTIES)} County\n"
    f"{len(aoi_fires_s2)} fires since 2015 | "
    f"{aoi_fires_s2['acres'].sum():,.0f} acres",
    fontsize=14, fontweight="bold", color="white",
)
ax.set_axis_off()

fig.patch.set_facecolor(BG_COLOR)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "chosen_aoi.png", dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

print(f"\n✓ AOI selected! Use the GEE coordinates from C9 in your Earth Engine scripts.")